In [3]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console

In [4]:
model = OpenAIChatCompletionClient(model="gpt-4o-mini")

clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client=model,
    system_message="""You are an expert editor focused on clarity and simplicity. 
            Your job is to eliminate ambiguity, redundancy, and make every sentence crisp and clear. 
            Don't worry about persuasion or tone — just make the message easy to read and understand.""",
)

tone_agent = AssistantAgent(
    "ToneAgent",
    model_client=model,
    system_message="""You are a communication coach focused on emotional tone and professionalism. 
            Your job is to make the email sound warm, confident, and human — while staying professional 
            and appropriate for the audience. Improve the emotional resonance, polish the phrasing, 
            and adjust any words that may come off as stiff, cold, or overly casual.""",
)

persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client=model,
    system_message="""You are a persuasion expert trained in marketing, behavioral psychology, 
            and copywriting. Your job is to enhance the email's persuasive power: improve call to action, structure arguments, and emphasize benefits. Remove weak or passive language.""",
)

synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client=model,
    system_message="""You are an advanced email-writing specialist. Your role is to read all 
            prior agent responses and revisions, and then **synthesize the best ideas** into a unified, 
            polished draft of the email. Focus on: Integrating clarity, tone, and persuasion improvements; 
            Ensuring coherence, fluency, and a natural voice; Creating a version that feels professional, 
            effective, and readable.""",
)

critic_agent = AssistantAgent(
    "CriticAgent",
    model_client=model,
    system_message="""You are an email quality evaluator. Your job is to perform a final review 
            of the synthesized email and determine if it meets professional standards. Review the email for: 
            Clarity and flow, appropriate professional tone, effective call-to-action, and overall coherence.
            Be constructive but decisive. If the email has major flaws (unclear message, unprofessional tone, 
            or missing key elements), provide ONE specific improvement suggestion. If the email meets professional standards and communicates effectively, respond with 'The email meets professional standards.' followed by `TERMINATE` on a new line. You should only approve emails that are perfect enough for professional use, dont settle.""",
)

In [6]:
text_termination =TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=30)

termiantion_condition = text_termination | max_messages_termination

In [7]:
team = RoundRobinGroupChat(
    participants=[
        clarity_agent,
        tone_agent,
        persuasion_agent,
        synthesizer_agent,
        critic_agent
    ],
    termination_condition=termiantion_condition,
)

await Console(
    # run은 모든 결과가 나온 다음 실패 메시지 등 확인 가능
    # run_stream은 실시간으로 실패 메시지 등 확인 가능
    team.run_stream(task="Hi! I'm hungry, buy me lunch and invest in my business. Thanks.")
)

---------- TextMessage (user) ----------
Hi! I'm hungry, buy me lunch and invest in my business. Thanks.
---------- TextMessage (ClarityAgent) ----------
I'm hungry. Please buy me lunch and invest in my business. Thank you.
---------- TextMessage (ToneAgent) ----------
Subject: A Little Help

Hi [Recipient's Name],

I hope this message finds you well! I'm reaching out because I'm feeling a bit hungry and was wondering if you might consider treating me to lunch. It would also be wonderful to discuss potential investment opportunities in my business, as I truly believe we could create something special together.

Thank you so much for your support and consideration!

Warm regards,  
[Your Name]
---------- TextMessage (PersuasionAgent) ----------
Subject: Let's Connect Over Lunch!

Hi [Recipient's Name],

I hope this message finds you in great spirits! I'm reaching out to share a unique opportunity that could benefit both of us. 

First, I'm feeling a bit hungry and would love to catch up

TaskResult(messages=[TextMessage(id='fc46f189-eb0e-4e82-8a19-c7751be0e761', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 2, 1, 5, 6, 24, 695331, tzinfo=datetime.timezone.utc), content="Hi! I'm hungry, buy me lunch and invest in my business. Thanks.", type='TextMessage'), TextMessage(id='2e6e3bae-e231-4245-a7cf-1ed2f4b82ac1', source='ClarityAgent', models_usage=RequestUsage(prompt_tokens=77, completion_tokens=16), metadata={}, created_at=datetime.datetime(2026, 2, 1, 5, 6, 26, 781333, tzinfo=datetime.timezone.utc), content="I'm hungry. Please buy me lunch and invest in my business. Thank you.", type='TextMessage'), TextMessage(id='e7bcfc10-fdbc-4486-9a1b-262c17f98479', source='ToneAgent', models_usage=RequestUsage(prompt_tokens=121, completion_tokens=84), metadata={}, created_at=datetime.datetime(2026, 2, 1, 5, 6, 29, 417389, tzinfo=datetime.timezone.utc), content="Subject: A Little Help\n\nHi [Recipient's Name],\n\nI hope this message finds you well